In [435]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

# directory that contains agent_tools.py (must run before importing agent_tools)
_pkg = Path.cwd()
# print(_pkg)
if (_pkg / "agent_tools.py").is_file():
    pkg = _pkg
else:
    pkg = _pkg.parent  # typical: cwd is .../notebook, module is one level up

sys.path.insert(0, str(pkg.resolve()))

# Drop stale cache so edits to agent_tools.py are picked up without restarting the kernel
sys.modules.pop("agent_tools", None)
import agent_tools as at
# from agent_tools import AgentMemory


In [436]:
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 300)

# Input Data
Loading dataframe from a parquet or csv data located in path

In [437]:
path_data = '..\data\heloc_dataset_v1.parquet'

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\arifn\AppData\Local\Temp\ipykernel_33252\1931592331.py:1: SyntaxWarning: invalid escape sequence '\d'
  path_data = '..\data\heloc_dataset_v1.parquet'


In [438]:
try:
    df = pd.read_parquet(path_data)
except:
    try:
        df = pd.read_csv(path_data)
    except:
        raise ValueError(f"Failed to load data from {path_data}")

# Metadata
Setting metadata for column aliases, target column, feature column, time related column, data type (for data split), etc. The purpose of this aliases is so that we don't need to manually change the column name one-by-obne if we want to use different column for label, features, etc.  

col_time: datetime of event. e.g, date of application, click date, etc.  
col_target: column name of groundtruth label.  
cols_feat: column name of features used for training.  
col_day: column name of day-truncated col_time.  
col_month: column name of month-truncated col_time.  
col_type: column name of data type, whether the data is for train, test, valid, or hold-out sample (hoot, oot).  
col_score: column name of predicted probability score.
cols_feat_num: column name of numerical features.  
cols_feat_cat: column name of categorical features.  
cols_feat_time: column name of datetime features.


In [439]:
col_time = 'date'
col_target = 'label'
cols_feat = ['ExternalRiskEstimate', 'MSinceOldestTradeOpen',
       'MSinceMostRecentTradeOpen', 'AverageMInFile', 'NumSatisfactoryTrades',
       'NumTrades60Ever2DerogPubRec', 'NumTrades90Ever2DerogPubRec',
       'PercentTradesNeverDelq', 'MSinceMostRecentDelq',
       'MaxDelq2PublicRecLast12M', 'MaxDelqEver', 'NumTotalTrades',
       'NumTradesOpeninLast12M', 'PercentInstallTrades',
       'MSinceMostRecentInqexcl7days', 'NumInqLast6M', 'NumInqLast6Mexcl7days',
       'NetFractionRevolvingBurden', 'NetFractionInstallBurden',
       'NumRevolvingTradesWBalance', 'NumInstallTradesWBalance',
       'NumBank2NatlTradesWHighUtilization', 'PercentTradesWBalance']
col_day = 'day'
col_month = 'month'
col_type = 'type'
col_score = 'score'

In [440]:
df[col_day] = at.get_day_from_date(df, col_time)
df[col_month] = at.get_month_from_date(df, col_time)
cols_feat_num, cols_feat_cat, cols_feat_time = at.get_feature_dtype(df, cols_feat)


# EDA
Explarotary Data Analysis: Understanding the data quality, feature distribution, and labels before start modelling.  
In this session we also do preliminary filtering on the data by making sure that:  
1. Features has enough penetration: Missing rate less than 80%

## Missing rate of features

In [441]:
df_nan_rate = at.get_nan_rate(df, cols_feat)
display(df_nan_rate)

,feature_name,missing_rate
0,ExternalRiskEstimate,0.0
1,MSinceOldestTradeOpen,0.0
2,MSinceMostRecentTradeOpen,0.0
3,AverageMInFile,0.0
4,NumSatisfactoryTrades,0.0
5,NumTrades60Ever2DerogPubRec,0.0
6,NumTrades90Ever2DerogPubRec,0.0
7,PercentTradesNeverDelq,0.0
8,MSinceMostRecentDelq,0.0
9,MaxDelq2PublicRecLast12M,0.0


## Missing rate of features over period of time.

In [442]:
df_nan_rate_timely = at.get_nan_rate_timely(df, cols_feat, col_month)
display(df_nan_rate_timely)

missing_rate                              \
time_period                              201501 201502 201503 201504 201505   
feature_name                                                                  
AverageMInFile                              0.0    0.0    0.0    0.0    0.0   
ExternalRiskEstimate                        0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentDelq                        0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentInqexcl7days                0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentTradeOpen                   0.0    0.0    0.0    0.0    0.0   
MSinceOldestTradeOpen                       0.0    0.0    0.0    0.0    0.0   
MaxDelq2PublicRecLast12M                    0.0    0.0    0.0    0.0    0.0   
MaxDelqEver                                 0.0    0.0    0.0    0.0    0.0   
NetFractionInstallBurden                    0.0    0.0    0.0    0.0    0.0   
NetFractionRevolvingBurden                  0.0    0.0    0.0    0.0    0.0   
NumBank2NatlTradesWHighUtilization          0.0    0.0    0.0    0.0    0.0   
NumInqLast6M                                0.0    0.0    0.0    0.0    0.0   
NumInqLast6Mexcl7days                       0.0    0.0    0.0    0.0    0.0   
NumInstallTradesWBalance                    0.0    0.0    0.0    0.0    0.0   
NumRevolvingTradesWBalance                  0.0    0.0    0.0    0.0    0.0   
NumSatisfactoryTrades                       0.0    0.0    0.0    0.0    0.0   
NumTotalTrades                              0.0    0.0    0.0    0.0    0.0   
NumTrades60Ever2DerogPubRec                 0.0    0.0    0.0    0.0    0.0   
NumTrades90Ever2DerogPubRec                 0.0    0.0    0.0    0.0    0.0   
NumTradesOpeninLast12M                      0.0    0.0    0.0    0.0    0.0   
PercentInstallTrades                        0.0    0.0    0.0    0.0    0.0   
PercentTradesNeverDelq                      0.0    0.0    0.0    0.0    0.0   
PercentTradesWBalance                       0.0    0.0    0.0    0.0    0.0   

                                                                              \
time_period                        201506 201507 201508 201509 201510 201511   
feature_name                                                                   
AverageMInFile                        0.0    0.0    0.0    0.0    0.0    0.0   
ExternalRiskEstimate                  0.0    0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentDelq                  0.0    0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentInqexcl7days          0.0    0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentTradeOpen             0.0    0.0    0.0    0.0    0.0    0.0   
MSinceOldestTradeOpen                 0.0    0.0    0.0    0.0    0.0    0.0   
MaxDelq2PublicRecLast12M              0.0    0.0    0.0    0.0    0.0    0.0   
MaxDelqEver                           0.0    0.0    0.0    0.0    0.0    0.0   
NetFractionInstallBurden              0.0    0.0    0.0    0.0    0.0    0.0   
NetFractionRevolvingBurden            0.0    0.0    0.0    0.0    0.0    0.0   
NumBank2NatlTradesWHighUtilization    0.0    0.0    0.0    0.0    0.0    0.0   
NumInqLast6M                          0.0    0.0    0.0    0.0    0.0    0.0   
NumInqLast6Mexcl7days                 0.0    0.0    0.0    0.0    0.0    0.0   
NumInstallTradesWBalance              0.0    0.0    0.0    0.0    0.0    0.0   
NumRevolvingTradesWBalance            0.0    0.0    0.0    0.0    0.0    0.0   
NumSatisfactoryTrades                 0.0    0.0    0.0    0.0    0.0    0.0   
NumTotalTrades                        0.0    0.0    0.0    0.0    0.0    0.0   
NumTrades60Ever2DerogPubRec           0.0    0.0    0.0    0.0    0.0    0.0   
NumTrades90Ever2DerogPubRec           0.0    0.0    0.0    0.0    0.0    0.0   
NumTradesOpeninLast12M                0.0    0.0    0.0    0.0    0.0    0.0   
PercentInstallTrades                  0.0    0.0    0.0    0.0    0.0    0.0   
PercentTradesNeverDelq                0.0    0.0    0.0    0

# Split Data

In this section we split data into several data type:  

oot: Out of time sample, to check the performance of the model after development data.  
hoot: Historical out of time sample, to check the performance of the model before the development data.    
train: data sample for training model.  
valid: data sample for hyperparameter tuning.  
test: data sample for on-paper evaluation. 
In Summary, data type must follow thiss equence of event: oot --> train (t-3) | test | valid (t-2)  --> oot (t-1), where t is time notation.  

By looking at:
1. Distribution of target rate over period of time.  
2. Data count over the period of time.  
3. Number of positive label over the period of time.  

In the case of limited data, we should prioritize the data split with the following order:  
1. train (for model building: should have minimum 1000 data count) and oot data (for performance evaluation: should have min 300 data count).  
2. test.  
3. valid.  
4. hoot.  

### Target (groundtruth) rate over period of time

In [443]:
df_stats_target_mean_per_month = at.get_timely_binary_target_rate(df, col_month, col_target)
display(df_stats_target_mean_per_month)

,mean,count,count_positive
month,,,
201501,0.480534,899,432
201502,0.396552,812,322
201503,0.353726,899,318
201504,0.480460,870,418
201505,0.583982,899,525
201506,0.489655,870,426
201507,0.474972,899,427
201508,0.527374,895,472
201509,0.386905,840,325


In [444]:
at.plot_timely_binary_target_rate(df, col_month, col_target, 'target_rate_per_month.png')

In [445]:
oot_th = 201510
hoot_th = 201503

In [446]:
# Use the same integer scale for `col_period` and for oot_th / hoot_th (here: `month`).
df[col_type] = at.split_data(df, col_target, col_month, oot_th, hoot_th)

In [447]:
df[col_type].value_counts()

type
train    2636
hoot     2610
oot      2576
test     1319
valid    1318
Name: count, dtype: int64

Checking the consistency of the label distribution in each sample type.

In [448]:
df_target_rate_per_sample = at.get_target_rate_sample(df, col_type, col_target)
display(df_target_rate_per_sample)


,count,mean
type,,
hoot,2610,0.410728
oot,2576,0.518245
test,1319,0.492039
train,2636,0.491654
valid,1318,0.491654


# Feature transformation
1. Transform the feature into Weight of Evidence to handle non-monotinicity and inbalanced label.  
2. Fine tune the WoE if necessary based on: number of data each bin, number of positive label each bin, number of positive rate (target mean) each bin.  

In [449]:
# Getting binningprocess object from otpbinning for woe transformation, woe statistics from training sample, and list of features that have issues during data transformation (missing feature, etc)
dict_bin, bp, df_binning_stats, binning_feature_issues = at.get_optimal_bin(
    df.loc[df[col_type] == "train"], cols_feat , col_target, cols_feat_cat=cols_feat_cat
)
display(df_binning_stats.sort_values(by='gini_power', ascending=False))

,name,dtype,status,selected,n_bins,iv,js,gini,quality_score,gini_power
11,NumTotalTrades,numerical,OPTIMAL,True,6,0.024596,0.003058,0.079225,0.002017,0.920775
2,MSinceMostRecentTradeOpen,numerical,OPTIMAL,True,6,0.035198,0.004382,0.100193,0.018778,0.899807
20,NumInstallTradesWBalance,numerical,OPTIMAL,True,6,0.042386,0.005271,0.108321,0.00329,0.891679
12,NumTradesOpeninLast12M,numerical,OPTIMAL,True,6,0.048713,0.006021,0.111746,0.010252,0.888254
4,NumSatisfactoryTrades,numerical,OPTIMAL,True,6,0.074214,0.009178,0.136911,0.064841,0.863089
6,NumTrades90Ever2DerogPubRec,numerical,OPTIMAL,True,3,0.126368,0.01552,0.143345,0.148225,0.856655
18,NetFractionInstallBurden,numerical,OPTIMAL,True,6,0.070341,0.008731,0.14392,0.031213,0.85608
19,NumRevolvingTradesWBalance,numerical,OPTIMAL,True,6,0.086865,0.010744,0.158586,0.160678,0.841414
16,NumInqLast6Mexcl7days,numerical,OPTIMAL,True,5,0.113001,0.013863,0.168369,0.106171,0.831631
15,NumInqLast6M,numerical,OPTIMAL,True,5,0.129148,0.015784,0.180347,0.059235,0.819653


In [450]:
# Getting binning table containing binning statistics for every feature. This statistics is used for fine-tuning if necessary.
df_binning_tables, features_issue_binning_tables = at.get_binning_tables_from_bp(bp, cols_feat)
display(df_binning_tables)

,Feature Name,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,ExternalRiskEstimate,"(-inf, 64.50)",676,0.256449,525,151,0.223373,1.212731,0.333839,0.039347
1,ExternalRiskEstimate,"[64.50, 68.50)",364,0.138088,246,118,0.324176,0.70126,0.064889,0.007949
2,ExternalRiskEstimate,"[68.50, 70.50)",170,0.064492,102,68,0.400000,0.372078,0.008800,0.001094
3,ExternalRiskEstimate,"[70.50, 74.50)",341,0.129363,175,166,0.486804,0.019411,0.000049,0.000006
4,ExternalRiskEstimate,"[74.50, 80.50)",450,0.170713,158,292,0.648889,-0.647546,0.069545,0.008544
5,ExternalRiskEstimate,"[80.50, inf)",635,0.240895,134,501,0.788976,-1.352153,0.387492,0.045054
6,ExternalRiskEstimate,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
7,ExternalRiskEstimate,Missing,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
8,ExternalRiskEstimate,,2636,1.000000,1340,1296,0.491654,,0.864614,0.101994
9,MSinceOldestTradeOpen,"(-inf, 31.50)",148,0.056146,69,79,0.533784,-0.168728,0.001597,0.000199


In [451]:
at.draw_binning_tables_from_bp(bp, cols_feat, "./binning_figures")

['F:\\datascience\\dsa-agent\\predictive-model-agent\\notebook\\binning_figures\\ExternalRiskEstimate_binning_table.png',
 'F:\\datascience\\dsa-agent\\predictive-model-agent\\notebook\\binning_figures\\MSinceOldestTradeOpen_binning_table.png',
 'F:\\datascience\\dsa-agent\\predictive-model-agent\\notebook\\binning_figures\\MSinceMostRecentTradeOpen_binning_table.png',
 'F:\\datascience\\dsa-agent\\predictive-model-agent\\notebook\\binning_figures\\AverageMInFile_binning_table.png',
 'F:\\datascience\\dsa-agent\\predictive-model-agent\\notebook\\binning_figures\\NumSatisfactoryTrades_binning_table.png',
 'F:\\datascience\\dsa-agent\\predictive-model-agent\\notebook\\binning_figures\\NumTrades60Ever2DerogPubRec_binning_table.png',
 'F:\\datascience\\dsa-agent\\predictive-model-agent\\notebook\\binning_figures\\NumTrades90Ever2DerogPubRec_binning_table.png',
 'F:\\datascience\\dsa-agent\\predictive-model-agent\\notebook\\binning_figures\\PercentTradesNeverDelq_binning_table.png',
 'F:\\d

In [452]:
dict_bin

{'ExternalRiskEstimate': [64.5, 68.5, 70.5, 74.5, 80.5],
 'MSinceOldestTradeOpen': [31.5, 107.5, 134.5, 212.5, 273.5],
 'MSinceMostRecentTradeOpen': [2.5, 3.5, 7.5, 9.5, 16.5],
 'AverageMInFile': [52.5, 65.5, 80.5, 97.5, 137.5],
 'NumSatisfactoryTrades': [5.5, 9.5, 11.5, 19.5, 39.5],
 'NumTrades60Ever2DerogPubRec': [0.5, 1.5, 2.5],
 'NumTrades90Ever2DerogPubRec': [0.5, 1.5],
 'PercentTradesNeverDelq': [63.5, 75.5, 83.5, 90.5, 95.5],
 'MSinceMostRecentDelq': [-3.5, 2.5, 15.5, 22.5, 46.5],
 'MaxDelq2PublicRecLast12M': [1.5, 5.5, 6.5],
 'MaxDelqEver': [3.5, 5.5, 6.5],
 'NumTotalTrades': [8.5, 11.5, 21.5, 24.5, 45.5],
 'NumTradesOpeninLast12M': [0.5, 1.5, 2.5, 3.5, 5.5],
 'PercentInstallTrades': [24.5, 35.5, 42.5, 46.5, 65.5],
 'MSinceMostRecentInqexcl7days': [-7.5, 0.5, 1.5, 7.5, 12.5],
 'NumInqLast6M': [0.5, 1.5, 2.5, 4.5],
 'NumInqLast6Mexcl7days': [0.5, 1.5, 2.5, 4.5],
 'NetFractionRevolvingBurden': [14.5, 26.5, 39.5, 65.5, 75.5],
 'NetFractionInstallBurden': [7.5, 33.5, 61.5, 72.5, 93

In [453]:
# Fine-tune WOE if necessary. If not, skip it.
dict_bin = {'ExternalRiskEstimate': [64.5, 68.5, 70.5, 74.5, 80.5],
 'MSinceOldestTradeOpen': [31.5, 107.5, 134.5, 212.5, 273.5],
 'MSinceMostRecentTradeOpen': [2.5, 3.5, 7.5, 9.5, 16.5],
 'AverageMInFile': [52.5, 65.5, 80.5, 97.5, 137.5],
 'NumSatisfactoryTrades': [5.5, 9.5, 11.5, 19.5, 39.5],
 'NumTrades60Ever2DerogPubRec': [0.5, 1.5, 2.5],
 'NumTrades90Ever2DerogPubRec': [0.5, 1.5],
 'PercentTradesNeverDelq': [63.5, 75.5, 83.5, 90.5, 95.5],
 'MSinceMostRecentDelq': [-3.5, 2.5, 15.5, 22.5, 46.5],
 'MaxDelq2PublicRecLast12M': [1.5, 5.5, 6.5],
 'MaxDelqEver': [3.5, 5.5, 6.5],
 'NumTotalTrades': [8.5, 11.5, 21.5, 24.5, 45.5],
 'NumTradesOpeninLast12M': [0.5, 1.5, 2.5, 3.5, 5.5],
 'PercentInstallTrades': [24.5, 35.5, 42.5, 46.5, 65.5],
 'MSinceMostRecentInqexcl7days': [-7.5, 0.5, 1.5, 7.5, 12.5],
 'NumInqLast6M': [0.5, 1.5, 2.5, 4.5],
 'NumInqLast6Mexcl7days': [0.5, 1.5, 2.5, 4.5],
 'NetFractionRevolvingBurden': [14.5, 26.5, 39.5, 65.5, 75.5],
 'NetFractionInstallBurden': [7.5, 33.5, 61.5, 72.5, 93.5],
 'NumRevolvingTradesWBalance': [1.5, 2.5, 4.5, 6.5, 7.5],
 'NumInstallTradesWBalance': [-3.5, 1.5, 2.5, 3.5, 4.5],
 'NumBank2NatlTradesWHighUtilization': [0.5, 1.5, 2.5, 3.5],
 'PercentTradesWBalance': [44.5, 64.5, 75.5, 84.5, 89.5]}

bp, df_binning_stats, binning_feature_issues = at.modify_optimal_bin(df, dict_bin, cols_feat, col_target, cols_feat_cat)

In [454]:
df_binning_stats

,name,dtype,status,selected,n_bins,iv,js,gini,quality_score,gini_power
0,ExternalRiskEstimate,numerical,OPTIMAL,True,6,0.855131,0.10065,0.486913,0.314658,0.513087
1,MSinceOldestTradeOpen,numerical,OPTIMAL,True,6,0.194453,0.023794,0.236757,0.646593,0.763243
2,MSinceMostRecentTradeOpen,numerical,INFEASIBLE,True,1,0.0,0.0,0,0.0,1
3,AverageMInFile,numerical,OPTIMAL,True,6,0.253459,0.031103,0.27737,0.488868,0.72263
4,NumSatisfactoryTrades,numerical,OPTIMAL,True,6,0.084848,0.010519,0.153702,0.152694,0.846298
5,NumTrades60Ever2DerogPubRec,numerical,OPTIMAL,True,4,0.164937,0.020238,0.182989,0.045416,0.817011
6,NumTrades90Ever2DerogPubRec,numerical,OPTIMAL,True,3,0.121237,0.014839,0.139113,0.214827,0.860887
7,PercentTradesNeverDelq,numerical,OPTIMAL,True,6,0.311017,0.037853,0.285256,0.66836,0.714744
8,MSinceMostRecentDelq,numerical,OPTIMAL,True,6,0.261788,0.032016,0.258724,0.627953,0.741276
9,MaxDelq2PublicRecLast12M,numerical,OPTIMAL,True,4,0.291842,0.035675,0.287697,0.774936,0.712303


In [455]:
# Getting transformed woe data, name of columns containing woe data, and list of features that have issues during data transformation (missing feature, etc)
X_woe, cols_feat_woe, features_issue_woe = at.get_woe_from_bp(df, cols_feat, bp)

In [456]:
# Transform cols_feat to woe value and store it as new columns in the existing dataframe 
df[cols_feat_woe] = bp.transform(df[cols_feat], metric="woe")
display(df[cols_feat_woe])

,ExternalRiskEstimate_woe,MSinceOldestTradeOpen_woe,MSinceMostRecentTradeOpen_woe,AverageMInFile_woe,NumSatisfactoryTrades_woe,NumTrades60Ever2DerogPubRec_woe,NumTrades90Ever2DerogPubRec_woe,PercentTradesNeverDelq_woe,MSinceMostRecentDelq_woe,MaxDelq2PublicRecLast12M_woe,MaxDelqEver_woe,NumTotalTrades_woe,NumTradesOpeninLast12M_woe,PercentInstallTrades_woe,MSinceMostRecentInqexcl7days_woe,NumInqLast6M_woe,NumInqLast6Mexcl7days_woe,NetFractionRevolvingBurden_woe,NetFractionInstallBurden_woe,NumRevolvingTradesWBalance_woe,NumInstallTradesWBalance_woe,NumBank2NatlTradesWHighUtilization_woe,PercentTradesWBalance_woe
0,1.054241,-0.044439,2.220446e-16,-0.349456,-0.217190,0.882663,-0.172234,0.988806,0.986278,0.887186,0.576366,-0.100174,-0.095739,0.073863,0.372190,-0.247742,-0.233822,-0.007279,2.220446e-16,2.220446e-16,-0.092359,0.211931,0.057473
1,1.054241,0.897530,2.220446e-16,0.752955,0.568278,0.882663,0.925487,-0.475878,-0.405128,0.382841,-0.498298,0.364563,-0.103332,0.876635,0.372190,-0.247742,-0.233822,-0.622259,2.220446e-16,2.220446e-16,-0.097490,-0.401003,-0.605843
2,0.702694,0.897530,2.220446e-16,0.752955,0.433947,-0.248388,-0.172234,-0.475878,-0.405128,-0.568373,-0.498298,0.218546,0.131259,0.073863,0.372190,0.315639,0.352801,0.503967,2.220446e-16,2.220446e-16,-0.049402,0.211931,0.641080
3,0.702694,-0.044439,2.220446e-16,-0.054815,-0.217190,0.465254,0.536113,0.234620,-0.195169,0.084648,0.307768,-0.200571,0.121846,0.402916,0.372190,0.784107,0.352801,0.886401,2.220446e-16,2.220446e-16,0.258321,0.821699,0.948188
4,-1.477371,-0.577968,2.220446e-16,-0.604722,0.012186,-0.248388,-0.172234,-0.475878,-0.405128,-0.568373,-0.498298,0.045621,-0.103332,-0.155215,0.372190,-0.027298,-0.011080,0.503967,2.220446e-16,2.220446e-16,-0.092359,-0.401003,0.365942
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10454,-0.036534,0.467000,2.220446e-16,0.258648,-0.217190,-0.248388,-0.172234,0.234620,-0.195169,0.084648,0.307768,0.045621,0.131259,-0.194113,-0.610937,-0.247742,-0.233822,-0.326912,2.220446e-16,2.220446e-16,-0.049402,-0.401003,0.948188
10455,0.702694,-0.044439,2.220446e-16,-0.054815,0.332151,-0.248388,-0.172234,0.234620,0.088063,0.084648,0.307768,0.045621,-0.103332,-0.077230,-0.224850,-0.027298,-0.011080,1.439089,2.220446e-16,2.220446e-16,-0.049402,0.211931,0.365942
10456,-0.036534,0.467000,2.220446e-16,0.258648,0.012186,0.465254,0.536113,-0.475878,-0.405128,0.084648,-0.498298,0.045621,-0.095739,-0.155215,-0.610937,0.315639,0.352801,-0.622259,2.220446e-16,2.220446e-16,-0.097490,-0.401003,-0.336695
10457,-0.036534,-0.280233,2.220446e-16,-0.604722,-0.399120,0.862088,0.925487,-0.475878,0.088063,0.084648,0.449788,-0.200571,-0.103332,-0.194113,-0.610937,-0.247742,-0.233822,-0.326912,2.220446e-16,2.220446e-16,-0.092359,-0.401003,-0.605843


# Feature Selection
In this section we implement different methodology for filtering the input feature so that only feature that has:
1. stability over time 
2. good performance  
can enter the model.

In [457]:
# Filtering for stability. We should as possible as we can minimize the target rate cross-over in feature bin over period of the time.
df_stats = at.get_timely_target_rate_feature_segment(df.loc[df[col_type]=='train'], cols_feat_woe, col_target, col_month)
display(df_stats)

,time period,feature name,segment,count data,count positive,positive rate
0,201504,ExternalRiskEstimate_woe,-1.477371,95,73,0.768421
1,201504,ExternalRiskEstimate_woe,-0.643091,74,47,0.635135
2,201504,ExternalRiskEstimate_woe,-0.036534,64,35,0.546875
3,201504,ExternalRiskEstimate_woe,0.414998,31,11,0.354839
4,201504,ExternalRiskEstimate_woe,0.702694,73,29,0.397260
...,...,...,...,...,...,...
661,201509,PercentTradesWBalance_woe,-0.336695,98,47,0.479592
662,201509,PercentTradesWBalance_woe,0.057473,82,34,0.414634
663,201509,PercentTradesWBalance_woe,0.365942,57,17,0.298246
664,201509,PercentTradesWBalance_woe,0.641080,31,9,0.290323


In [458]:
# Pick only the desired features.
cols_feat_woe = at.set_value_list(['ExternalRiskEstimate_woe',
 'MSinceOldestTradeOpen_woe',
 'MSinceMostRecentTradeOpen_woe',
 'AverageMInFile_woe',
 'NumSatisfactoryTrades_woe',
 'NumTrades60Ever2DerogPubRec_woe',
 'NumTrades90Ever2DerogPubRec_woe',
 'PercentTradesNeverDelq_woe',
 'MSinceMostRecentDelq_woe',
 'MaxDelq2PublicRecLast12M_woe',
 'MaxDelqEver_woe',
 'NumTotalTrades_woe',
 'NumTradesOpeninLast12M_woe',
 'PercentInstallTrades_woe',
 'MSinceMostRecentInqexcl7days_woe',
 'NumInqLast6M_woe',
 'NumInqLast6Mexcl7days_woe',
 'NetFractionRevolvingBurden_woe',
 'NetFractionInstallBurden_woe',
 'NumRevolvingTradesWBalance_woe',
 'NumInstallTradesWBalance_woe',
 'NumBank2NatlTradesWHighUtilization_woe',
 'PercentTradesWBalance_woe'])

In [459]:

selected_feats_auc, feats_stats_auc = at.select_features_auc_max_corr(df.loc[df[col_type]=='train'], cols_feat_woe, col_target, max_corr=0.5)
display(feats_stats_auc)

,feature_name,max_correlation,auc_roc,standardized_auc_roc
0,ExternalRiskEstimate_woe,0.498863,0.252408,0.747592
1,PercentTradesWBalance_woe,0.406580,0.334096,0.665904
2,AverageMInFile_woe,0.341112,0.351952,0.648048
3,MSinceMostRecentInqexcl7days_woe,0.299108,0.372215,0.627785
4,NumBank2NatlTradesWHighUtilization_woe,0.498863,0.376375,0.623625
5,NumTrades60Ever2DerogPubRec_woe,0.442341,0.402545,0.597455
6,PercentInstallTrades_woe,0.390458,0.403380,0.596620
7,NumInqLast6M_woe,0.302512,0.409826,0.590174
8,NumSatisfactoryTrades_woe,0.348837,0.431545,0.568455
9,NumTradesOpeninLast12M_woe,0.302512,0.444127,0.555873


In [460]:
selected_feats_iv, feats_stats_iv = at.select_features_iv_max_corr(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)
display(feats_stats_iv)

,feature_name,max_correlation,iv,auc_roc,standardized_auc_roc
0,ExternalRiskEstimate_woe,0.498863,0.809552,0.252408,0.747592
1,PercentTradesWBalance_woe,0.406580,0.342842,0.334096,0.665904
2,MSinceMostRecentInqexcl7days_woe,0.299108,0.315523,0.372215,0.627785
3,AverageMInFile_woe,0.341112,0.286387,0.351952,0.648048
4,NumBank2NatlTradesWHighUtilization_woe,0.498863,0.150515,0.376375,0.623625
5,PercentInstallTrades_woe,0.390458,0.120870,0.403380,0.596620
6,NumTrades60Ever2DerogPubRec_woe,0.442341,0.101090,0.402545,0.597455
7,NumInqLast6M_woe,0.302512,0.093200,0.409826,0.590174
8,NumSatisfactoryTrades_woe,0.348837,0.069875,0.431545,0.568455
9,NumTradesOpeninLast12M_woe,0.302512,0.034169,0.444127,0.555873


In [461]:
selected_feats_aic_fwd, df_stats_aic_fwd = at.select_features_aic_forward(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)
display(df_stats_aic_fwd)

,step_number,added_feature_name,max_correlation,total_aic,delta_aic,roc_auc,gini
0,1,ExternalRiskEstimate_woe,0.000000,3122.544488,-532.992968,0.747592,0.495183
1,2,MSinceMostRecentInqexcl7days_woe,0.201530,3024.026068,-98.518419,0.771246,0.542492
2,3,PercentInstallTrades_woe,0.116182,2980.714645,-43.311423,0.781527,0.563054
3,4,MSinceOldestTradeOpen_woe,0.249673,2950.311292,-30.403353,0.786934,0.573868
4,5,PercentTradesWBalance_woe,0.406580,2937.466934,-12.844358,0.789942,0.579885
5,6,NumInstallTradesWBalance_woe,0.296758,2933.624595,-3.842339,0.791047,0.582093
6,7,NumBank2NatlTradesWHighUtilization_woe,0.498863,2932.460023,-1.164572,0.791784,0.583569
7,8,NumSatisfactoryTrades_woe,0.348837,2930.525260,-1.934762,0.792278,0.584555
8,9,NumTradesOpeninLast12M_woe,0.295831,2928.608004,-1.917257,0.793069,0.586139
9,10,NumTrades60Ever2DerogPubRec_woe,0.442341,2927.417300,-1.190703,0.793857,0.587713


In [462]:
selected_feats_aic_bwd, df_stats_aic_bwd = at.select_features_aic_backward(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)
display(df_stats_aic_bwd)

,step_number,eliminated_feature_name,max_correlation,total_aic,delta_aic,roc_auc,gini
0,1,NumInqLast6M_woe,0.498863,2927.4173,-0.052366,0.793857,0.587713


In [463]:
selected_feats_bic_bwd, df_stats_bic_bwd = at.select_features_bic_backward(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)
display(df_stats_bic_bwd)

,step_number,eliminated_feature_name,max_correlation,total_bic,delta_bic,roc_auc,gini
0,1,NumInqLast6M_woe,0.498863,2992.064497,-5.929384,0.793857,0.587713
1,2,NumInstallTradesWBalance_woe,0.498863,2987.329377,-4.735121,0.793008,0.586016
2,3,NumTrades60Ever2DerogPubRec_woe,0.498863,2982.862460,-4.466916,0.792571,0.585142
3,4,NumTradesOpeninLast12M_woe,0.498863,2977.751511,-5.110949,0.791842,0.583685
4,5,NumBank2NatlTradesWHighUtilization_woe,0.406580,2975.537858,-2.213654,0.790937,0.581874
5,6,NumSatisfactoryTrades_woe,0.406580,2972.729041,-2.808816,0.789942,0.579885


In [464]:
selected_feats_bic_fwd, df_stats_bic_fwd = at.select_features_bic_forward(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)
display(selected_feats_bic_fwd)

['ExternalRiskEstimate_woe',
 'MSinceMostRecentInqexcl7days_woe',
 'PercentInstallTrades_woe',
 'MSinceOldestTradeOpen_woe',
 'PercentTradesWBalance_woe']

In [465]:
selected_feats_auc_fwd, df_stats_auc_fwd = at.select_features_auc_forward(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)
display(selected_feats_auc_fwd)

['ExternalRiskEstimate_woe',
 'MSinceMostRecentInqexcl7days_woe',
 'PercentInstallTrades_woe',
 'MSinceOldestTradeOpen_woe']

# Model training

In [466]:
dict_statistics_auc, best_model_auc, importance_auc = at.train_logreg_l2_tune_cv(df, selected_feats_auc, col_target, col_type)
dict_statistics_iv, best_model_iv, importance_iv = at.train_logreg_l2_tune_cv(df, selected_feats_iv, col_target, col_type)
dict_statistics_aic_fwd, best_model_aic_fwd, importance_aic_fwd = at.train_logreg_l2_tune_cv(df, selected_feats_aic_fwd, col_target, col_type)
dict_statistics_aic_bwd, best_model_aic_bwd, importance_aic_bwd = at.train_logreg_l2_tune_cv(df, selected_feats_aic_bwd, col_target, col_type)
dict_statistics_bic_fwd, best_model_bic_fwd, importance_bic_fwd = at.train_logreg_l2_tune_cv(df, selected_feats_bic_fwd, col_target, col_type)
dict_statistics_bic_bwd, best_model_bic_bwd, importance_bic_bwd = at.train_logreg_l2_tune_cv(df, selected_feats_bic_bwd, col_target, col_type)
dict_statistics_auc_fwd, best_model_auc_fwd, importance_auc_fwd = at.train_logreg_l2_tune_cv(df, selected_feats_auc_fwd, col_target, col_type)

In [467]:
best_model_iv.get_params()

{'C': 0.1,
 'class_weight': None,
 'dual': False,
 'fit_intercept': True,
 'intercept_scaling': 1,
 'l1_ratio': 0.0,
 'max_iter': 10000,
 'n_jobs': None,
 'penalty': 'deprecated',
 'random_state': 42,
 'solver': 'lbfgs',
 'tol': 0.0001,
 'verbose': 0,
 'warm_start': False}

# Model Prediction

In [468]:
df['proba_0_auc'], df['proba_1_auc'], feature_predict_issue = at.logreg_predict(df, best_model_auc, cols_feat_woe)
df['proba_0_iv'], df['proba_1_iv'], feature_predict_issue = at.logreg_predict(df, best_model_iv, cols_feat_woe)
df['proba_0_aic_fwd'], df['proba_1_aic_fwd'], feature_predict_issue = at.logreg_predict(df, best_model_aic_fwd, cols_feat_woe)
df['proba_0_aic_bwd'], df['proba_1_aic_bwd'], feature_predict_issue = at.logreg_predict(df, best_model_bic_fwd, cols_feat_woe)
df['proba_0_bic_fwd'], df['proba_1_bic_fwd'], feature_predict_issue = at.logreg_predict(df, best_model_aic_bwd, cols_feat_woe)
df['proba_0_bic_bwd'], df['proba_1_bic_bwd'], feature_predict_issue = at.logreg_predict(df, best_model_bic_bwd, cols_feat_woe)
df['proba_0_auc_fwd'], df['proba_1_auc_fwd'], feature_predict_issue = at.logreg_predict(df, best_model_auc_fwd, cols_feat_woe)

# Model Evaluation

In [469]:
cols_score = ['proba_1_auc', 'proba_1_iv', 'proba_1_aic_fwd', 'proba_1_aic_bwd', 'proba_1_bic_fwd', 'proba_1_bic_bwd', 'proba_1_auc_fwd']
df_stats_performance_per_scores = at.compare_scores_gini_auc(df, col_type, cols_score, col_target)
display(df_stats_performance_per_scores.transpose())

,proba_1_auc,proba_1_iv,proba_1_aic_fwd,proba_1_aic_bwd,proba_1_bic_fwd,proba_1_bic_bwd,proba_1_auc_fwd
gini_hoot,0.478653,0.484264,0.468612,0.459594,0.464801,0.455199,0.459130
gini_oot,0.522408,0.525259,0.513757,0.501277,0.509625,0.500315,0.498786
gini_test,0.520951,0.520045,0.517933,0.511736,0.495943,0.513115,0.507410
gini_train,0.532984,0.532763,0.536692,0.523927,0.507080,0.523695,0.520999
gini_valid,0.503934,0.501820,0.505270,0.495509,0.478165,0.497241,0.493659
auc_hoot,0.739327,0.742132,0.734306,0.729797,0.732400,0.727600,0.729565
auc_oot,0.761204,0.762630,0.756879,0.750638,0.754812,0.750157,0.749393
auc_test,0.760475,0.760022,0.758967,0.755868,0.747972,0.756558,0.753705
auc_train,0.766492,0.766381,0.768346,0.761963,0.753540,0.761848,0.760500
auc_valid,0.751967,0.750910,0.752635,0.747754,0.739082,0.748620,0.746830


In [470]:
df_stats_score_psi_over_time = at.get_timely_vars_psi(df.loc[df[col_type]=='train'], df.loc[df[col_type]!='train'], cols_score, col_month, n_bins=5)
display(df_stats_score_psi_over_time)

,proba_1_auc,proba_1_iv,proba_1_aic_fwd,proba_1_aic_bwd,proba_1_bic_fwd,proba_1_bic_bwd,proba_1_auc_fwd
time,,,,,,,
201501,0.007439,0.006492,0.005713,0.007352,0.010119,0.005813,0.009791
201502,0.590324,0.630399,0.591272,0.623866,0.638507,0.622457,0.627754
201503,0.040736,0.049013,0.053389,0.047161,0.050568,0.047020,0.041007
201504,0.019267,0.008732,0.014151,0.011403,0.021067,0.013919,0.023651
201505,0.041501,0.038088,0.049701,0.058162,0.028908,0.051932,0.049801
201506,0.012863,0.023017,0.016213,0.008366,0.017971,0.008897,0.016720
201507,0.016983,0.015013,0.016221,0.021147,0.016441,0.023441,0.021139
201508,0.007960,0.004396,0.005173,0.005636,0.008977,0.004208,0.007942
201509,0.029092,0.033943,0.033758,0.036466,0.025619,0.040431,0.040575


# Deep dive individual score
After we chose the score we want to use, we can do further analysis as the final check.

In [471]:
df_stats = at.get_score_predictive_power_data_type(df, 'proba_1_iv', col_type, col_target)
display(df_stats)

,time_period,aucroc,gini,count,count_positive,positive_rate
0,hoot,0.742132,0.484264,2610,1072,0.410728
1,oot,0.762630,0.525259,2576,1335,0.518245
2,test,0.760022,0.520045,1319,649,0.492039
3,train,0.766381,0.532763,2636,1296,0.491654
4,valid,0.750910,0.501820,1318,648,0.491654


In [472]:
df_stats = at.get_score_predictive_power_timely(df, 'proba_1_iv', col_month, col_target)
display(df_stats)

,time_period,aucroc,gini,count,count_positive,positive_rate
0,201501,0.744223,0.488446,899,432,0.480534
1,201502,0.698460,0.396920,812,322,0.396552
2,201503,0.772232,0.544464,899,318,0.353726
3,201504,0.730287,0.460574,870,418,0.480460
4,201505,0.769756,0.539511,899,525,0.583982
5,201506,0.745366,0.490732,870,426,0.489655
6,201507,0.786523,0.573046,899,427,0.474972
7,201508,0.817877,0.635753,895,472,0.527374
8,201509,0.692128,0.384257,840,325,0.386905
9,201510,0.734342,0.468684,868,435,0.501152


In [473]:
df_stats = at.get_score_predictive_power_data_type_bootstrap(df, 'proba_1_iv', col_type, col_target)

In [474]:
df_stats

,time_period,CI_5_gini,mean_gini,CI_95_gini,CI_5_aucroc,mean_auc,CI_95_aucroc,mean_positive_rate
0,hoot,0.312608,0.486931,0.650033,0.656304,0.743465,0.825016,0.41057
1,oot,0.360849,0.525988,0.675982,0.680425,0.762994,0.837991,0.51935
2,test,0.359194,0.517316,0.667965,0.679597,0.758658,0.833982,0.49123
3,train,0.382123,0.532160,0.675651,0.691061,0.766080,0.837826,0.49279
4,valid,0.336508,0.502923,0.658685,0.668254,0.751462,0.829343,0.49158


In [475]:
df_stats = at.compare_score_predictive_power_data_type_bootstrap(df, 'proba_1_iv', 'proba_1_auc', col_type, col_target)

In [476]:
df_stats

,time_period,champion_CI_5_gini,champion_mean_gini,champion_CI_95_gini,challenger_CI_5_gini,challenger_mean_gini,challenger_CI_95_gini,champion_CI_5_aucroc,champion_mean_auc,champion_CI_95_aucroc,challenger_CI_5_aucroc,challenger_mean_auc,challenger_CI_95_aucroc,mean_positive_rate
0,hoot,0.312608,0.486931,0.650033,0.304716,0.480346,0.648803,0.656304,0.743465,0.825016,0.652358,0.740173,0.824402,0.41057
1,oot,0.348880,0.522059,0.669530,0.346534,0.518862,0.667555,0.674440,0.761030,0.834765,0.673267,0.759431,0.833777,0.51786
2,test,0.353463,0.518004,0.662453,0.359250,0.519429,0.666112,0.676731,0.759002,0.831227,0.679625,0.759714,0.833056,0.49160
3,train,0.370675,0.531033,0.676454,0.366136,0.531707,0.676306,0.685338,0.765517,0.838227,0.683068,0.765854,0.838153,0.49267
4,valid,0.332361,0.502887,0.653186,0.338025,0.505373,0.657909,0.666181,0.751444,0.826593,0.669012,0.752686,0.828954,0.49252


In [477]:
df_stats = at.get_timely_feature_psi_woe(df.loc[df[col_type]=='train'], df.loc[df[col_type]!='train'], cols_feat_woe, col_month)

In [478]:
df_stats

,time,feature_name,psi,count data
0,201501,ExternalRiskEstimate_woe,0.001409,899
1,201501,MSinceOldestTradeOpen_woe,0.003847,899
2,201501,MSinceMostRecentTradeOpen_woe,0.000000,899
3,201501,AverageMInFile_woe,0.009819,899
4,201501,NumSatisfactoryTrades_woe,0.009620,899
5,201501,NumTrades60Ever2DerogPubRec_woe,0.004625,899
6,201501,NumTrades90Ever2DerogPubRec_woe,0.005780,899
7,201501,PercentTradesNeverDelq_woe,0.002614,899
8,201501,MSinceMostRecentDelq_woe,0.006516,899
9,201501,MaxDelq2PublicRecLast12M_woe,0.005187,899


In [479]:
# df_stats = at.get_timely_psi(df.loc[df[col_type]=='train'], df.loc[df[col_type]!='train'], 'proba_1', col_month)

In [480]:
# df_stats

In [481]:
model_params = best_model_iv.get_params()

In [482]:
df_stats, scorecard = at.create_scorecard_model(df.loc[df[col_type]=='train'], cols_feat, model_params, col_target, bp)

In [483]:
display(df_stats)

,Variable,Bin id,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS,Coefficient,Points
0,ExternalRiskEstimate,0,"(-inf, 62.50)",522,0.198027,406,116,0.222222,1.219376,2.603110e-01,3.066187e-02,-0.270697,30.718233
1,ExternalRiskEstimate,1,"[62.50, 64.50)",154,0.058422,119,35,0.227273,1.190388,7.356576e-02,8.688551e-03,-0.270697,30.491821
2,ExternalRiskEstimate,2,"[64.50, 68.50)",364,0.138088,246,118,0.324176,0.701260,6.488948e-02,7.948966e-03,-0.270697,26.671397
3,ExternalRiskEstimate,3,"[68.50, 70.50)",170,0.064492,102,68,0.400000,0.372078,8.799746e-03,1.093667e-03,-0.270697,24.100265
4,ExternalRiskEstimate,4,"[70.50, 72.50)",181,0.068665,93,88,0.486188,0.021876,3.285179e-05,4.106392e-06,-0.270697,21.364949
5,ExternalRiskEstimate,5,"[72.50, 74.50)",160,0.060698,82,78,0.487500,0.016623,1.677043e-05,2.096280e-06,-0.270697,21.323925
6,ExternalRiskEstimate,6,"[74.50, 76.50)",156,0.059181,61,95,0.608974,-0.476390,1.323415e-02,1.638802e-03,-0.270697,17.473158
7,ExternalRiskEstimate,7,"[76.50, 80.50)",294,0.111533,97,197,0.670068,-0.741880,5.906707e-02,7.218589e-03,-0.270697,15.399504
8,ExternalRiskEstimate,8,"[80.50, 84.50)",255,0.096737,61,194,0.760784,-1.190371,1.239998e-01,1.464513e-02,-0.270697,11.896483
9,ExternalRiskEstimate,9,"[84.50, 87.50)",212,0.080425,42,170,0.801887,-1.431516,1.429076e-01,1.647913e-02,-0.270697,10.012982


In [484]:
at.writre_production_code(bp, scorecard, "scoring.py")

'F:\\datascience\\dsa-agent\\predictive-model-agent\\notebook\\scoring.py'

In [485]:
df.iloc[0][cols_feat].to_dict()

{'ExternalRiskEstimate': 55,
 'MSinceOldestTradeOpen': 144,
 'MSinceMostRecentTradeOpen': 4,
 'AverageMInFile': 84,
 'NumSatisfactoryTrades': 20,
 'NumTrades60Ever2DerogPubRec': 3,
 'NumTrades90Ever2DerogPubRec': 0,
 'PercentTradesNeverDelq': 83,
 'MSinceMostRecentDelq': 2,
 'MaxDelq2PublicRecLast12M': 3,
 'MaxDelqEver': 5,
 'NumTotalTrades': 23,
 'NumTradesOpeninLast12M': 1,
 'PercentInstallTrades': 43,
 'MSinceMostRecentInqexcl7days': 0,
 'NumInqLast6M': 0,
 'NumInqLast6Mexcl7days': 0,
 'NetFractionRevolvingBurden': 33,
 'NetFractionInstallBurden': -8,
 'NumRevolvingTradesWBalance': 8,
 'NumInstallTradesWBalance': 1,
 'NumBank2NatlTradesWHighUtilization': 1,
 'PercentTradesWBalance': 69}

In [486]:
df[col_score] = -scorecard.score(df[cols_feat])

In [487]:
at.scorecard_plot_cap(df.loc[df[col_type]!='train'], col_target, col_score, scorecard, 'cap.png')

In [488]:
at.scorecard_plot_auc_roc(df.loc[df[col_type]!='train'], col_target, col_score, scorecard, 'auc_roc.png')

In [489]:
at.scorecard_plot_ks(df.loc[df[col_type]!='train'], col_target, col_score, scorecard, 'ks.png')

In [490]:
from scoring import get_woe, get_score


dict_feat = {'ExternalRiskEstimate': 55,
 'MSinceOldestTradeOpen': 144,
 'MSinceMostRecentTradeOpen': 4,
 'AverageMInFile': 84,
 'NumSatisfactoryTrades': 20,
 'NumTrades60Ever2DerogPubRec': 3,
 'NumTrades90Ever2DerogPubRec': 0,
 'PercentTradesNeverDelq': 83,
 'MSinceMostRecentDelq': 2,
 'MaxDelq2PublicRecLast12M': 3,
 'MaxDelqEver': 5,
 'NumTotalTrades': 23,
 'NumTradesOpeninLast12M': 1,
 'PercentInstallTrades': 43,
 'MSinceMostRecentInqexcl7days': 0,
 'NumInqLast6M': 0,
 'NumInqLast6Mexcl7days': 0,
 'NetFractionRevolvingBurden': 33,
 'NetFractionInstallBurden': -8,
 'NumRevolvingTradesWBalance': 8,
 'NumInstallTradesWBalance': 1,
 'NumBank2NatlTradesWHighUtilization': 1,
 'PercentTradesWBalance': 69}

In [491]:
get_woe(dict_feat)

{'ExternalRiskEstimate': 1.219375952462631,
 'MSinceOldestTradeOpen': 0.055166381308707976,
 'MSinceMostRecentTradeOpen': 0.06857791062395104,
 'AverageMInFile': -0.2847014443136433,
 'NumSatisfactoryTrades': -0.11120403838954482,
 'NumTrades60Ever2DerogPubRec': 0.9825335567902594,
 'NumTrades90Ever2DerogPubRec': -0.1799432881288305,
 'PercentTradesNeverDelq': 0.9802320598019799,
 'MSinceMostRecentDelq': 1.1452679803089092,
 'MaxDelq2PublicRecLast12M': 0.9591187666408247,
 'MaxDelqEver': 0.6643368315546199,
 'NumTotalTrades': -0.009856518622543019,
 'NumTradesOpeninLast12M': -0.033387016032737055,
 'PercentInstallTrades': 0.07783861907748735,
 'MSinceMostRecentInqexcl7days': 0.42614531334570316,
 'NumInqLast6M': -0.3361573881357198,
 'NumInqLast6Mexcl7days': -0.29397579782899186,
 'NetFractionRevolvingBurden': 0.20403322163815557,
 'NetFractionInstallBurden': -0.21258844549044806,
 'NumRevolvingTradesWBalance': 0.6384827660799233,
 'NumInstallTradesWBalance': -0.08347112043014834,
 'Nu

In [492]:
get_score(dict_feat)

{'score': 541.4297875603456}